# ML Tutor — Week 5: Feature engineering & validation
**Track:** Core ML · **Lab:** Lab 5 — Improve readmission prediction without cheating

**Objectives this week:**
1. Engineer features: encoding, scaling, binning, and interactions. *(CO5)*
2. Apply cross-validation and recognize/prevent data leakage. *(CO5)*
3. Improve a readmission-prediction model through iterative feature work. *(CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


**Dataset:** Synthetic inpatient discharge dataset with a binary 30-day readmission label and candidate predictors such as age_band, comorbidity_count, prior_admits_6m, length_of_stay, discharge_service, med_count, payer_type, weekend_discharge, and one intentionally suspicious leakage-style column for students to reject. No PHI.


## 1. Build a clean baseline validation loop
Create a baseline classification pipeline for readmission and evaluate it with cross-validation rather than one lucky split.

**Checkpoint:** Checkpoint 1 verifies: cross-validation is stratified, transforms live inside the pipeline, and the reported metrics come from cross_validate rather than a single holdout only.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

target = "readmitted_30d"

# readmit_df is already loaded in the notebook.
X = ...
y = ...

categorical_cols = ["age_band", "discharge_service", "payer_type", "weekend_discharge"]
numeric_cols = ["comorbidity_count", "prior_admits_6m", "length_of_stay", "med_count"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: run cross_validate with at least recall and roc_auc
scores = ...


## 2. Engineer one sensible feature change
Add one justified feature transformation such as binning, an interaction, or a derived rate, then compare cross-validated results with the baseline.

**Checkpoint:** Checkpoint 2 verifies: exactly one engineered feature is added, it is described in plain language, and students compare mean cross-validated metrics rather than cherry-picking a single fold.


In [ ]:
# Example idea placeholders — choose ONE and justify it.
# Option A: bucket length_of_stay into short / medium / long
# Option B: create an interaction like comorbidity_count * med_count
# Option C: transform prior_admits_6m into a simpler high-utilization flag

feature_df = readmit_df.copy()

# TODO: create your engineered feature here
feature_df["engineered_feature"] = ...

# TODO: rebuild X from feature_df and compare the engineered pipeline vs baseline
...


## 3. Catch and remove the leakage feature
Inspect candidate columns and exclude the one that should not be available at prediction time.

**Checkpoint:** Checkpoint 3 verifies: the suspicious future-information column is excluded, and the notebook contains a one- to two-sentence explanation tying leakage to prediction timing rather than to “bad correlation.”


In [ ]:
candidate_cols = [
    "age_band",
    "comorbidity_count",
    "prior_admits_6m",
    "length_of_stay",
    "discharge_service",
    "med_count",
    "payer_type",
    "weekend_discharge",
    "post_discharge_followup_completed",
]

# TODO: choose which column is leakage-risk and explain why
safe_cols = ...

# TODO: rebuild X using only safe_cols and rerun your validation
...


## 4. Hold out the final test set for one honest final look
After your feature decisions are settled with cross-validation, make one train/test split and report final test metrics with a short reflection on stability.

**Checkpoint:** Checkpoint 4 verifies: the test split happens after feature decisions are finalized, and the written reflection compares CV expectations with final test behavior instead of claiming certainty from one run.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, roc_auc_score

# TODO: create X_final and y_final from your leakage-safe feature set
X_final = ...
y_final = ...

X_train, X_test, y_train, y_test = ...

# TODO: fit your chosen pipeline on X_train, y_train
# TODO: compute test recall and test AUROC
...


## Homework — HW5 — Honest improvement plan for readmission prediction
Using the teaching readmission dataset, compare a baseline pipeline with one carefully engineered variant. Use cross-validation for model selection, reject at least one leakage-risk feature explicitly, then run one final holdout evaluation. Focus on explaining why each feature choice was allowed or disallowed.

**Deliverable:** One runnable notebook plus a 300-word reflection. The reflection must name one engineered feature you kept, one tempting feature you rejected as leakage, and whether the final test result matched your cross-validation expectations.


In [ ]:
# HW workspace

